In [1]:
import os, json, time, random
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, roc_curve, auc

import torch
import pickle
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

def load_compact_pkl_dataset(dataset_path,dataset_name):
    with open(dataset_path+dataset_name+'.pkl','rb') as f:
        dataset = pickle.load(f)
    return dataset

# =========================
# You provide these:
# =========================
dataset_name = 'ManySig'
dataset_path = '../ManySig.pkl/'
compact_dataset = load_compact_pkl_dataset(dataset_path, dataset_name)

# =========================
# Experiment config
# =========================
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Use 6 TX total; for open-set you must reserve some as unknown.
KNOWN_TX_NUM = 4   # 4 known + 2 unknown (recommended for 6-TX setting)
MAX_SIG_PER_TRIPLE = None  # per (tx,rx,day,eq) max signals; None = use all

# Choose source day and source RX
TRAIN_DATES = ['2021_03_15']  # 1 day as source domain
TEST_DATES_RX = ['2021_03_15']  # for cross-RX drift (same day)
TEST_DATES_DAY = ['2021_03_01'] # for cross-day drift (same RX)

# Feature sources
Z_FROM_EQ = 1       # classifier input uses equalized=1 or unequalized=0
D_FROM = "raw"      # "raw" or "eq"  (domain descriptor source)

# Training hyperparams
BATCH_SIZE = 128
EPOCHS = 60
LR = 1e-4
WEIGHT_DECAY = 1e-3
PATIENCE = 8

IN_PLANES = 64
DROPOUT = 0.3
EMB_DIM = 512

# Domain descriptor dimension
D_DIM = 32

# Block split for CV (reduce correlation leakage)
BLOCK_SIZE = 200  # within each (tx,rx,day) group, form blocks of 200 sequential signals

# Output dir
ts = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = f"./results_wisig_module2/run_{ts}"
os.makedirs(RUN_DIR, exist_ok=True)

config = dict(
    dataset_name=dataset_name,
    TRAIN_DATES=TRAIN_DATES,
    TEST_DATES_RX=TEST_DATES_RX,
    TEST_DATES_DAY=TEST_DATES_DAY,
    KNOWN_TX_NUM=KNOWN_TX_NUM,
    MAX_SIG_PER_TRIPLE=MAX_SIG_PER_TRIPLE,
    Z_FROM_EQ=Z_FROM_EQ,
    D_FROM=D_FROM,
    BATCH_SIZE=BATCH_SIZE,
    EPOCHS=EPOCHS,
    LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY,
    PATIENCE=PATIENCE,
    BLOCK_SIZE=BLOCK_SIZE,
    D_DIM=D_DIM,
)
with open(os.path.join(RUN_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)
print("RUN_DIR:", RUN_DIR)

Device: cuda
RUN_DIR: ./results_wisig_module2/run_20260304_095459


In [2]:
def get_idx(lst, val):
    return lst.index(val)

def get_signals(compact_dataset, tx, rx, date, equalized):
    """
    Return np.ndarray shape (N,256,2), dtype float32
    """
    tx_i = get_idx(compact_dataset["tx_list"], tx)
    rx_i = get_idx(compact_dataset["rx_list"], rx)
    date_i = get_idx(compact_dataset["capture_date_list"], date)
    eq_i = get_idx(compact_dataset["equalized_list"], equalized)
    sigs = compact_dataset["data"][tx_i][rx_i][date_i][eq_i]  # sequence of (256,2)
    sigs = np.array(sigs, dtype=np.float32)
    return sigs

def subsample(sigs, max_sig):
    if max_sig is None:
        return sigs
    return sigs[:min(len(sigs), max_sig)]

def iq_to_complex(x_256_2):
    # x: (...,256,2) -> (...,256) complex64
    return (x_256_2[...,0] + 1j*x_256_2[...,1]).astype(np.complex64)

In [3]:
def get_idx(lst, val):
    return lst.index(val)

def get_signals(compact_dataset, tx, rx, date, equalized):
    """
    Return np.ndarray shape (N,256,2), dtype float32
    """
    tx_i = get_idx(compact_dataset["tx_list"], tx)
    rx_i = get_idx(compact_dataset["rx_list"], rx)
    date_i = get_idx(compact_dataset["capture_date_list"], date)
    eq_i = get_idx(compact_dataset["equalized_list"], equalized)
    sigs = compact_dataset["data"][tx_i][rx_i][date_i][eq_i]  # sequence of (256,2)
    sigs = np.array(sigs, dtype=np.float32)
    return sigs

def subsample(sigs, max_sig):
    if max_sig is None:
        return sigs
    return sigs[:min(len(sigs), max_sig)]

def iq_to_complex(x_256_2):
    # x: (...,256,2) -> (...,256) complex64
    return (x_256_2[...,0] + 1j*x_256_2[...,1]).astype(np.complex64)

In [4]:
def domain_feat_256_complex(x256_c, d_dim=32):
    """
    x256_c: (256,) complex64
    return: (d_dim,) float32
    """
    x = x256_c / (np.sqrt(np.mean(np.abs(x256_c)**2)) + 1e-12)
    X = np.fft.fft(x, n=256)
    mag = np.abs(X) + 1e-12
    logm = np.log(mag)
    D = np.fft.rfft(logm, n=512)
    feat = np.real(D[:d_dim]).astype(np.float32)
    return feat

def compute_domain_feats_batch(X_256_2, d_from="raw"):
    """
    X_256_2: (N,256,2) float32
    return: (N,D_DIM) float32
    """
    Xc = iq_to_complex(X_256_2)
    feats = np.stack([domain_feat_256_complex(Xc[i], d_dim=D_DIM) for i in range(Xc.shape[0])], axis=0)
    return feats

In [ ]:
tx_list = compact_dataset['tx_list']
rx_list = compact_dataset['rx_list']
date_list = compact_dataset['capture_date_list']

print("TX:", len(tx_list), tx_list)
print("RX:", len(rx_list), rx_list)
print("DATE:", len(date_list), date_list)

# Choose 6 TX, 12 RX, 4 DAY
TX_USE = tx_list[:6]
RX_USE = rx_list[:12]
DATE_USE = date_list[:4]

KNOWN_TX = TX_USE[:KNOWN_TX_NUM]
UNKNOWN_TX = TX_USE[KNOWN_TX_NUM:]

# =========================
# NEW: randomly choose X source RXs using SEED
# =========================
SOURCE_RX_NUM = 9  # <-- 你想随机选几个source RX，就改这里（X）
rng = np.random.RandomState(SEED)

if SOURCE_RX_NUM < 1 or SOURCE_RX_NUM > len(RX_USE):
    raise ValueError(f"SOURCE_RX_NUM must be in [1, {len(RX_USE)}], got {SOURCE_RX_NUM}")

SOURCE_RXS = list(rng.choice(RX_USE, size=SOURCE_RX_NUM, replace=False))
SOURCE_RXS.sort()  # 可选：排序便于打印与复现实验

DRIFT_RX = [r for r in RX_USE if r not in SOURCE_RXS]

print("KNOWN_TX:", KNOWN_TX)
print("UNKNOWN_TX:", UNKNOWN_TX)
print("SOURCE_RXS:", SOURCE_RXS)
print("DRIFT_RX:", DRIFT_RX)

TX: 6 ['14-10', '14-7', '20-15', '20-19', '6-15', '8-20']
RX: 12 ['1-1', '1-19', '14-7', '18-2', '19-2', '2-1', '2-19', '20-1', '3-19', '7-14', '7-7', '8-8']
DATE: 4 ['2021_03_01', '2021_03_08', '2021_03_15', '2021_03_23']
KNOWN_TX: ['14-10', '14-7', '20-15', '20-19']
UNKNOWN_TX: ['6-15', '8-20']
SOURCE_RXS: ['1-1', '7-14', '7-7']
DRIFT_RX: ['1-19', '14-7', '18-2', '19-2', '2-1', '2-19', '20-1', '3-19', '8-8']


In [6]:
def build_source_train(compact_dataset):
    X_list, y_list, D_list = [], [], []
    group_keys = []  # for block CV: (tx, rx, day, block)

    for tx in KNOWN_TX:
        for rx in SOURCE_RXS:
            for day in TRAIN_DATES:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                if D_FROM == "raw":
                    Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0), MAX_SIG_PER_TRIPLE)
                else:
                    Xd = subsample(get_signals(compact_dataset, tx, rx, day, 1), MAX_SIG_PER_TRIPLE)

                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]
                Xd = Xd[:n]

                y = np.full((n,), KNOWN_TX.index(tx), dtype=np.int64)
                Df = compute_domain_feats_batch(Xd, d_from=D_FROM)

                X_list.append(Xz)
                y_list.append(y)
                D_list.append(Df)

                # block key within this (tx,rx,day)
                blk = (np.arange(n) // BLOCK_SIZE).astype(np.int64)
                # make global key integer
                base = (KNOWN_TX.index(tx) * 10_000_000) + (0 * 1_000_000) + (TRAIN_DATES.index(day) * 100_000)
                gk = base + blk
                group_keys.append(gk)

    X = np.concatenate(X_list, axis=0).astype(np.float32)   # (N,256,2)
    y = np.concatenate(y_list, axis=0).astype(np.int64)     # (N,)
    D = np.concatenate(D_list, axis=0).astype(np.float32)   # (N,D_DIM)
    G = np.concatenate(group_keys, axis=0).astype(np.int64) # (N,) block id

    return X, y, D, G

X_src, y_src, D_src, G_src = build_source_train(compact_dataset)
print("Source:", X_src.shape, y_src.shape, D_src.shape, G_src.shape)

Source: (12000, 256, 2) (12000,) (12000, 32) (12000,)


In [7]:
def build_eval_set(compact_dataset, txs, rxs, days, y_unknown=-1):
    Xz_list, y_list, D_list = [], [], []

    for tx in txs:
        for rx in rxs:
            for day in days:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                if D_FROM == "raw":
                    Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0), MAX_SIG_PER_TRIPLE)
                else:
                    Xd = subsample(get_signals(compact_dataset, tx, rx, day, 1), MAX_SIG_PER_TRIPLE)

                n = min(len(Xz), len(Xd))
                Xz = Xz[:n].astype(np.float32)
                Xd = Xd[:n].astype(np.float32)

                if tx in KNOWN_TX:
                    y = np.full((n,), KNOWN_TX.index(tx), dtype=np.int64)
                else:
                    y = np.full((n,), y_unknown, dtype=np.int64)

                Df = compute_domain_feats_batch(Xd, d_from=D_FROM)

                Xz_list.append(Xz); y_list.append(y); D_list.append(Df)

    if len(Xz_list) == 0:
        return None, None, None
    X = np.concatenate(Xz_list, axis=0)
    y = np.concatenate(y_list, axis=0)
    D = np.concatenate(D_list, axis=0)
    return X, y, D

X_drift_rx, y_drift_rx, D_drift_rx = build_eval_set(compact_dataset, KNOWN_TX, DRIFT_RX, TEST_DATES_RX)
X_unk_in, y_unk_in, D_unk_in       = build_eval_set(compact_dataset, UNKNOWN_TX, SOURCE_RXS, TEST_DATES_RX)
X_unk_drift, y_unk_drift, D_unk_drift = build_eval_set(compact_dataset, UNKNOWN_TX, DRIFT_RX, TEST_DATES_RX)
X_drift_day, y_drift_day, D_drift_day = build_eval_set(compact_dataset, KNOWN_TX, SOURCE_RXS, TEST_DATES_DAY)

print("Known drift (RX):", X_drift_rx.shape)
print("Unknown in-domain:", X_unk_in.shape)
print("Unknown + drift:", X_unk_drift.shape)
print("Known drift (DAY):", X_drift_day.shape)

Known drift (RX): (36000, 256, 2)
Unknown in-domain: (6000, 256, 2)
Unknown + drift: (18000, 256, 2)
Known drift (DAY): (12000, 256, 2)


In [8]:
class ArrayDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return self.X[i], self.y[i]

def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    criterion = nn.CrossEntropyLoss()
    total_loss, total_correct, total = 0.0, 0, 0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        if train:
            optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += float(loss.item()) * yb.size(0)
        pred = logits.argmax(dim=1)
        total_correct += int((pred == yb).sum().item())
        total += int(yb.size(0))
    return total_loss / max(1,total), total_correct / max(1,total)

def compute_embeddings(model, X_np, batch=512):
    model.eval()
    ds = ArrayDataset(X_np, np.zeros((X_np.shape[0],), dtype=np.int64))
    dl = DataLoader(ds, batch_size=batch, shuffle=False)
    Z = []
    with torch.no_grad():
        for xb, _ in dl:
            xb = xb.to(device)
            _, emb = model(xb, return_embed=True)
            Z.append(emb.cpu().numpy().astype(np.float32))
    return np.concatenate(Z, axis=0)

def l2norm(x, axis=1, eps=1e-12):
    return x / (np.linalg.norm(x, axis=axis, keepdims=True) + eps)

def prototypes(Z, y, K):
    ZN = l2norm(Z, axis=1)
    P = np.zeros((K, Z.shape[1]), dtype=np.float32)
    for k in range(K):
        idx = np.where(y==k)[0]
        P[k] = ZN[idx].mean(axis=0)
    return l2norm(P, axis=1)

def domain_stats(D, y, K, reg=1e-3):
    mu = np.zeros((K, D.shape[1]), dtype=np.float32)
    Sinv = np.zeros((K, D.shape[1], D.shape[1]), dtype=np.float32)
    for k in range(K):
        X = D[y==k].astype(np.float32)
        mu[k] = X.mean(axis=0)
        C = np.cov(X.T, bias=False).astype(np.float32)
        C = C + reg*np.eye(C.shape[0], dtype=np.float32)
        Sinv[k] = np.linalg.inv(C)
    return mu, Sinv

def cosine_to_proto(Z, P):
    ZN = l2norm(Z, axis=1)
    return ZN @ P.T

def mahalanobis_batch(D, mu, Sinv):
    X = D - mu.reshape(1,-1)
    return np.einsum("nd,dd,nd->n", X, Sinv, X).astype(np.float32)

def double_scores(Z, D, P, mu_dom, Sinv_dom):
    cos = cosine_to_proto(Z, P)          # (N,K)
    khat = np.argmax(cos, axis=1).astype(np.int64)
    Sid = np.max(cos, axis=1).astype(np.float32)
    Sdom = np.zeros((Z.shape[0],), dtype=np.float32)
    for k in range(P.shape[0]):
        idx = np.where(khat==k)[0]
        if idx.size:
            Sdom[idx] = mahalanobis_batch(D[idx], mu_dom[k], Sinv_dom[k])
    return Sid, khat, Sdom

In [9]:
class BasicBlock1D(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1, downsample=None, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv1d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=dropout)
        self.conv2 = nn.Conv1d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(planes)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        out = out + identity
        return self.relu(out)

class ResNet18_1D(nn.Module):
    def __init__(self, num_classes, in_planes=64, dropout=0.0):
        super().__init__()
        self.in_planes = in_planes
        self.conv1 = nn.Conv1d(2, in_planes, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(in_planes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(64, 2, stride=1, dropout=dropout)
        self.layer2 = self._make_layer(128, 2, stride=2, dropout=dropout)
        self.layer3 = self._make_layer(256, 2, stride=2, dropout=dropout)
        self.layer4 = self._make_layer(512, 2, stride=2, dropout=dropout)

        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, planes, blocks, stride, dropout):
        downsample = None
        if stride != 1 or self.in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv1d(self.in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(planes)
            )
        layers = [BasicBlock1D(self.in_planes, planes, stride, downsample, dropout)]
        self.in_planes = planes
        for _ in range(1, blocks):
            layers.append(BasicBlock1D(self.in_planes, planes, dropout=dropout))
        return nn.Sequential(*layers)

    def forward(self, x, return_embed=False):
        # x: (B, 256, 2) -> (B,2,256)
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).squeeze(-1)  # (B,512)
        logits = self.fc(x)
        if return_embed:
            return logits, x
        return logits

In [10]:
def save_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)

def plot_curves(hist, out_png):
    e = np.arange(1, len(hist["train_loss"])+1)
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(e, hist["train_loss"], label="train")
    plt.plot(e, hist["val_loss"], label="val")
    plt.grid(True); plt.legend(); plt.xlabel("epoch"); plt.ylabel("loss")
    plt.subplot(1,2,2)
    plt.plot(e, hist["train_acc"], label="train")
    plt.plot(e, hist["val_acc"], label="val")
    plt.grid(True); plt.legend(); plt.xlabel("epoch"); plt.ylabel("acc")
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()

def plot_confmat(cm, out_png, title):
    plt.figure(figsize=(5,4))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    plt.xlabel("pred"); plt.ylabel("true")
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()

def plot_hist(a, b, la, lb, title, out_png, bins=80):
    plt.figure(figsize=(6,4))
    plt.hist(a, bins=bins, density=True, alpha=0.6, label=la)
    plt.hist(b, bins=bins, density=True, alpha=0.6, label=lb)
    plt.grid(True); plt.legend()
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()

def plot_roc(y_true, score, title, out_png):
    fpr, tpr, _ = roc_curve(y_true, score)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(5,4))
    plt.plot(fpr, tpr, label=f"AUC={roc_auc:.3f}")
    plt.plot([0,1],[0,1],"--")
    plt.grid(True); plt.legend()
    plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()
    return float(roc_auc)

def plot_scatter(Sid, Sdom, lab, title, out_png, max_points=20000):
    n = len(Sid)
    if n > max_points:
        idx = np.random.choice(n, size=max_points, replace=False)
    else:
        idx = np.arange(n)
    plt.figure(figsize=(6,5))
    plt.scatter(Sid[idx], Sdom[idx], s=3, c=lab[idx], alpha=0.5)
    plt.grid(True)
    plt.xlabel("S_id (max cosine)")
    plt.ylabel("S_dom (Mahalanobis)")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()

In [11]:
# =========================
# t-SNE visualization utils
# =========================
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

def _sample_indices(n, max_n, rng):
    if max_n is None or max_n <= 0 or n <= max_n:
        return np.arange(n)
    return rng.choice(n, size=max_n, replace=False)

def tsne_fit_transform(Z, seed=42, pca_dim=50, perplexity=30):
    """Z: (N, D)"""
    Z = Z.astype(np.float32)
    if Z.shape[0] < 10:
        raise ValueError(f"Too few points for t-SNE: N={Z.shape[0]}")
    # PCA pre-reduction for speed/stability
    d = min(pca_dim, Z.shape[1], Z.shape[0]-1)
    Zp = PCA(n_components=d, random_state=seed).fit_transform(Z)
    # Perplexity must be < N
    perp = min(perplexity, max(5, (Z.shape[0]-1)//3))
    tsne = TSNE(
        n_components=2,
        perplexity=perp,
        init="pca",
        learning_rate="auto",
        max_iter=1200,
        random_state=seed,
        verbose=0,
    )
    Y = tsne.fit_transform(Zp)
    return Y, perp, d

def plot_tsne(Y, labels, label_names, out_png, title, max_points=None, seed=42):
    """Y: (N,2), labels: (N,) int"""
    rng = np.random.RandomState(seed)
    N = Y.shape[0]
    if max_points is not None and max_points > 0 and N > max_points:
        idx = rng.choice(N, size=max_points, replace=False)
        Y = Y[idx]
        labels = labels[idx]

    plt.figure(figsize=(7,6))
    unique = np.unique(labels)
    for lab in unique:
        pts = Y[labels == lab]
        name = label_names.get(int(lab), str(lab))
        plt.scatter(pts[:,0], pts[:,1], s=6, alpha=0.55, label=name)

    plt.title(title)
    plt.grid(True)
    plt.legend(markerscale=2, fontsize=9, loc="best")
    plt.tight_layout()
    plt.savefig(out_png, dpi=160)
    plt.close()

def make_tsne_scenario_plots(fold_dir, Z_A, Z_B, Z_C, Z_D, Z_E=None,
                            max_per_group=4000, seed=42):
    """Scenario plots to visualize confusion between unknown and drift."""
    rng = np.random.RandomState(seed)

    # Plot 1: A (stable known) vs B (known drift RX) vs C (unknown in-domain) vs D (unknown+drift)
    groups = [
        ("Known stable (A)", Z_A, 0),
        ("Known drift RX (B)", Z_B, 1),
        ("Unknown in-domain (C)", Z_C, 2),
        ("Unknown + drift (D)", Z_D, 3),
    ]

    Z_list, y_list = [], []
    for name, Zg, lab in groups:
        idx = _sample_indices(Zg.shape[0], max_per_group, rng)
        Z_list.append(Zg[idx])
        y_list.append(np.full((len(idx),), lab, dtype=np.int64))

    Z_all = np.concatenate(Z_list, axis=0)
    y_all = np.concatenate(y_list, axis=0)

    Y2d, perp, pca_d = tsne_fit_transform(Z_all, seed=seed, pca_dim=50, perplexity=30)
    plot_tsne(
        Y2d, y_all,
        {0:"Known stable",1:"Known drift (RX)",2:"Unknown",3:"Unknown+drift"},
        os.path.join(fold_dir, "tsne_scenarios_RX_unknown.png"),
        title=f"t-SNE (PCA{pca_d}, perp={perp}): Known vs Drift vs Unknown",
        max_points=20000,
        seed=seed
    )

    np.savez_compressed(
        os.path.join(fold_dir, "tsne_scenarios_RX_unknown.npz"),
        Y=Y2d, labels=y_all
    )

    # Plot 2: Cross-day drift (optional): A vs E (known drift day) vs C (unknown in-domain)
    if Z_E is not None:
        groups2 = [
            ("Known stable (A)", Z_A, 0),
            ("Known drift DAY (E)", Z_E, 4),
            ("Unknown in-domain (C)", Z_C, 2),
        ]
        Z_list, y_list = [], []
        for name, Zg, lab in groups2:
            idx = _sample_indices(Zg.shape[0], max_per_group, rng)
            Z_list.append(Zg[idx])
            y_list.append(np.full((len(idx),), lab, dtype=np.int64))
        Z_all = np.concatenate(Z_list, axis=0)
        y_all = np.concatenate(y_list, axis=0)

        Y2d, perp, pca_d = tsne_fit_transform(Z_all, seed=seed+1, pca_dim=50, perplexity=30)
        plot_tsne(
            Y2d, y_all,
            {0:"Known stable",4:"Known drift (DAY)",2:"Unknown"},
            os.path.join(fold_dir, "tsne_stable_day_unknown.png"),
            title=f"t-SNE (PCA{pca_d}, perp={perp}): Stable vs Day-drift vs Unknown",
            max_points=20000,
            seed=seed+1
        )
        np.savez_compressed(
            os.path.join(fold_dir, "tsne_stable_day_unknown.npz"),
            Y=Y2d, labels=y_all
        )


In [12]:
# Build block IDs for CV (use G_src)
# We need block-level stratification label -> device label
unique_blocks = np.unique(G_src)
block_label = np.zeros((len(unique_blocks),), dtype=np.int64)
for i,b in enumerate(unique_blocks):
    idx = np.where(G_src == b)[0]
    block_label[i] = y_src[idx[0]]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

K = len(KNOWN_TX)
fold_summ = []

for fold, (btr, bte) in enumerate(skf.split(unique_blocks, block_label), start=1):
    fold_dir = os.path.join(RUN_DIR, f"fold_{fold}")
    os.makedirs(fold_dir, exist_ok=True)

    tr_blocks = set(unique_blocks[btr])
    te_blocks = set(unique_blocks[bte])

    idx_tr = np.where(np.isin(G_src, list(tr_blocks)))[0]
    idx_te = np.where(np.isin(G_src, list(te_blocks)))[0]

    # val split from train indices
    rng = np.random.RandomState(SEED + fold)
    perm = rng.permutation(idx_tr)
    n_val = max(1, int(0.1 * len(perm)))
    idx_val = perm[:n_val]
    idx_train = perm[n_val:]

    X_train, y_train = X_src[idx_train], y_src[idx_train]
    X_val, y_val = X_src[idx_val], y_src[idx_val]
    X_test, y_test = X_src[idx_te], y_src[idx_te]   # in-domain (same rx/day) stable

    train_loader = DataLoader(ArrayDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(ArrayDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(ArrayDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

    model = ResNet18_1D(num_classes=K, in_planes=IN_PLANES, dropout=DROPOUT).to(device)
    opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_val = -1.0
    best_state = None
    patience = 0
    hist = {"train_loss":[], "train_acc":[], "val_loss":[], "val_acc":[]}

    for ep in range(1, EPOCHS+1):
        tr_loss, tr_acc = run_epoch(model, train_loader, optimizer=opt)
        va_loss, va_acc = run_epoch(model, val_loader, optimizer=None)

        hist["train_loss"].append(tr_loss)
        hist["train_acc"].append(tr_acc)
        hist["val_loss"].append(va_loss)
        hist["val_acc"].append(va_acc)

        if va_acc > best_val + 1e-4:
            best_val = va_acc
            best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
            if patience >= PATIENCE:
                break

    torch.save(best_state, os.path.join(fold_dir, "best_model.pth"))
    save_json(os.path.join(fold_dir, "history.json"), hist)
    plot_curves(hist, os.path.join(fold_dir, "train_curves.png"))

    model.load_state_dict(best_state)

    # In-domain closed-set accuracy
    model.eval()
    y_pred = []
    with torch.no_grad():
        for xb, _yb in test_loader:
            xb = xb.to(device)
            logits = model(xb)
            y_pred.append(logits.argmax(dim=1).cpu().numpy())
    y_pred = np.concatenate(y_pred)
    acc_in = float((y_pred == y_test).mean())
    cm = confusion_matrix(y_test, y_pred, labels=list(range(K)))
    plot_confmat(cm, os.path.join(fold_dir, "confmat_in_domain.png"), "In-domain Confusion")

    # Prototypes + domain stats from TRAIN split
    Z_tr = compute_embeddings(model, X_src[idx_train], batch=512)
    P = prototypes(Z_tr, y_src[idx_train], K)
    mu_dom, Sinv_dom = domain_stats(D_src[idx_train], y_src[idx_train], K, reg=1e-3)

    np.savez_compressed(os.path.join(fold_dir, "proto_dom_stats.npz"), P=P, mu_dom=mu_dom, Sinv_dom=Sinv_dom)

    # Stable scores on held-out blocks (A)
    Z_A = compute_embeddings(model, X_test, batch=512)
    Sid_A, khat_A, Sdom_A = double_scores(Z_A, D_src[idx_te], P, mu_dom, Sinv_dom)

    # Known-drift cross-RX (B)
    Z_B = compute_embeddings(model, X_drift_rx, batch=512)
    Sid_B, khat_B, Sdom_B = double_scores(Z_B, D_drift_rx, P, mu_dom, Sinv_dom)

    # Unknown in-domain (C)
    Z_C = compute_embeddings(model, X_unk_in, batch=512)
    Sid_C, khat_C, Sdom_C = double_scores(Z_C, D_unk_in, P, mu_dom, Sinv_dom)

    # Unknown + drift (D)
    Z_D = compute_embeddings(model, X_unk_drift, batch=512)
    Sid_D, khat_D, Sdom_D = double_scores(Z_D, D_unk_drift, P, mu_dom, Sinv_dom)

    # Cross-day drift (E)
    Z_E = compute_embeddings(model, X_drift_day, batch=512)
    Sid_E, khat_E, Sdom_E = double_scores(Z_E, D_drift_day, P, mu_dom, Sinv_dom)

    # Save raw scores
    np.savez_compressed(os.path.join(fold_dir, "scores_A_stable.npz"), Sid=Sid_A, Sdom=Sdom_A, y=y_test)
    np.savez_compressed(os.path.join(fold_dir, "scores_B_drift_rx.npz"), Sid=Sid_B, Sdom=Sdom_B, y=y_drift_rx)
    np.savez_compressed(os.path.join(fold_dir, "scores_C_unknown_in.npz"), Sid=Sid_C, Sdom=Sdom_C, y=y_unk_in)
    np.savez_compressed(os.path.join(fold_dir, "scores_D_unknown_drift.npz"), Sid=Sid_D, Sdom=Sdom_D, y=y_unk_drift)
    np.savez_compressed(os.path.join(fold_dir, "scores_E_drift_day.npz"), Sid=Sid_E, Sdom=Sdom_E, y=y_drift_day)

    # Plots: hist + ROC + scatter
    plot_hist(Sid_A, Sid_C, "Known(stable)", "Unknown(in-domain)", "S_id: Known vs Unknown",
              os.path.join(fold_dir, "hist_Sid_known_vs_unknown.png"))

    plot_hist(Sdom_A, Sdom_B, "Stable", "Known-Drift(cross-RX)", "S_dom: Stable vs Drift(RX)",
              os.path.join(fold_dir, "hist_Sdom_stable_vs_drift_rx.png"))

    plot_hist(Sdom_A, Sdom_E, "Stable", "Known-Drift(cross-day)", "S_dom: Stable vs Drift(DAY)",
              os.path.join(fold_dir, "hist_Sdom_stable_vs_drift_day.png"))

    # Unknown ROC using -Sid (unknown=1)
    y_u = np.concatenate([np.zeros_like(Sid_A, dtype=np.int64), np.ones_like(Sid_C, dtype=np.int64)])
    s_u = np.concatenate([-Sid_A, -Sid_C])
    auc_u = plot_roc(y_u, s_u, "Unknown ROC (score=-S_id)", os.path.join(fold_dir, "roc_unknown.png"))

    # Drift ROC using Sdom (drift=1): stable vs drift_rx
    y_dr = np.concatenate([np.zeros_like(Sdom_A, dtype=np.int64), np.ones_like(Sdom_B, dtype=np.int64)])
    s_dr = np.concatenate([Sdom_A, Sdom_B])
    auc_dr = plot_roc(y_dr, s_dr, "Drift ROC (cross-RX) (score=S_dom)", os.path.join(fold_dir, "roc_drift_rx.png"))

    # Drift ROC using Sdom: stable vs drift_day
    y_dd = np.concatenate([np.zeros_like(Sdom_A, dtype=np.int64), np.ones_like(Sdom_E, dtype=np.int64)])
    s_dd = np.concatenate([Sdom_A, Sdom_E])
    auc_dd = plot_roc(y_dd, s_dd, "Drift ROC (cross-day) (score=S_dom)", os.path.join(fold_dir, "roc_drift_day.png"))

    # Scatter Sid vs Sdom (4 scenarios)
    Sid_all = np.concatenate([Sid_A, Sid_B, Sid_C, Sid_D])
    Sdom_all = np.concatenate([Sdom_A, Sdom_B, Sdom_C, Sdom_D])
    lab = np.concatenate([
        np.zeros_like(Sid_A, dtype=np.int64),
        np.ones_like(Sid_B, dtype=np.int64),
        np.full_like(Sid_C, 2, dtype=np.int64),
        np.full_like(Sid_D, 3, dtype=np.int64),
    ])
    plot_scatter(Sid_all, Sdom_all, lab,
                 "S_id vs S_dom (0=stable,1=drift,2=unknown,3=unknown+drift)",
                 os.path.join(fold_dir, "scatter_Sid_Sdom.png"))

    # t-SNE visualization (scenario-level)
    try:
        make_tsne_scenario_plots(
            fold_dir, Z_A, Z_B, Z_C, Z_D, Z_E=Z_E,
            max_per_group=4000, seed=SEED + fold
        )
    except Exception as e:
        print(f"[Fold {fold}] t-SNE skipped: {e}")

    # Operating point thresholds for combined diagnosis (optional)
    tau_id = float(np.quantile(Sid_A, 0.05))    # 5% FRR on stable
    tau_dom = float(np.quantile(Sdom_A, 0.95))  # 5% false drift on stable

    def diagnose(Sid, Sdom):
        pred = np.zeros_like(Sid, dtype=np.int64)  # 0 stable
        pred[Sid < tau_id] = 2                     # 2 unknown
        pred[(Sid >= tau_id) & (Sdom > tau_dom)] = 1  # 1 drift
        return pred

    gt_A = np.zeros_like(Sid_A, dtype=np.int64)
    gt_B = np.ones_like(Sid_B, dtype=np.int64)
    gt_C = np.full_like(Sid_C, 2, dtype=np.int64)
    gt_D = np.full_like(Sid_D, 2, dtype=np.int64)  # unknown 优先

    pr = np.concatenate([diagnose(Sid_A,Sdom_A), diagnose(Sid_B,Sdom_B),
                         diagnose(Sid_C,Sdom_C), diagnose(Sid_D,Sdom_D)])
    gt = np.concatenate([gt_A, gt_B, gt_C, gt_D])
    cm3 = confusion_matrix(gt, pr, labels=[0,1,2])
    plot_confmat(cm3, os.path.join(fold_dir, "confmat_combined_diag.png"),
                 "Combined diag (0 stable,1 drift,2 unknown)")

    metrics = dict(
        fold=fold,
        acc_in_domain=acc_in,
        auc_unknown=auc_u,
        auc_drift_rx=auc_dr,
        auc_drift_day=auc_dd,
        tau_id=tau_id,
        tau_dom=tau_dom
    )
    save_json(os.path.join(fold_dir, "metrics.json"), metrics)
    fold_summ.append(metrics)

    print(f"[Fold {fold}] acc={acc_in:.3f}  AUC_u={auc_u:.3f}  AUC_drRX={auc_dr:.3f}  AUC_drDAY={auc_dd:.3f}")

save_json(os.path.join(RUN_DIR, "metrics_all_folds.json"), fold_summ)
print("Mean acc:", np.mean([m["acc_in_domain"] for m in fold_summ]))
print("Mean AUC unknown:", np.mean([m["auc_unknown"] for m in fold_summ]))
print("Mean AUC drift_rx:", np.mean([m["auc_drift_rx"] for m in fold_summ]))
print("Mean AUC drift_day:", np.mean([m["auc_drift_day"] for m in fold_summ]))


[Fold 1] acc=0.994  AUC_u=0.947  AUC_drRX=0.799  AUC_drDAY=0.616
[Fold 2] acc=0.999  AUC_u=0.933  AUC_drRX=0.780  AUC_drDAY=0.620
[Fold 3] acc=0.996  AUC_u=0.961  AUC_drRX=0.772  AUC_drDAY=0.623
[Fold 4] acc=0.997  AUC_u=0.981  AUC_drRX=0.804  AUC_drDAY=0.638
[Fold 5] acc=0.998  AUC_u=0.953  AUC_drRX=0.800  AUC_drDAY=0.620
Mean acc: 0.9966666666666667
Mean AUC unknown: 0.955081375
Mean AUC drift_rx: 0.7912019745370371
Mean AUC drift_day: 0.6231157916666666
